# PneumoNet — Baseline CNN
Custom 3-layer CNN for pneumonia detection from chest X-rays.

**Setup (run once):**
```bash
pip install -r ../requirements.txt
```
Place your dataset at `chest_xray/` in the project root (or update `config.yaml`).

In [ ]:
import sys
sys.path.insert(0, '..')

import yaml
import torch
import torch.nn as nn
import torch.optim as optim
from src.data_loader import get_dataloaders
from src.models import PneumoNetCNN, load_model
from src.train import train_loop
from src.evaluate import evaluate, plot_confusion_matrix, plot_roc_curve, plot_training_curves
from src.explain import find_last_conv, show_gradcam, show_shap, show_lime

with open('../config.yaml') as f:
    cfg = yaml.safe_load(f)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 1. Load Dataset

In [ ]:
train_loader, val_loader, test_loader, train_data, val_data, test_data = get_dataloaders(
    dataset_path=cfg['dataset']['path'],
    batch_size=cfg['dataset']['batch_size'],
    num_workers=cfg['dataset']['num_workers'],
    image_size=cfg['dataset']['image_size'],
)
print(f'Train: {len(train_data)} | Val: {len(val_data)} | Test: {len(test_data)}')
print('Classes:', train_data.classes)

## 2. Build Model

In [ ]:
model = PneumoNetCNN().to(device)
print(model)

## 3. Train
Best model (by val accuracy) is saved to `cfg['models']['cnn_path']`.

In [ ]:
cfg_cnn   = cfg['training']['cnn']
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=cfg_cnn['lr'])

history = train_loop(
    model, train_loader, val_loader, criterion, optimizer,
    num_epochs=cfg_cnn['epochs'],
    device=device,
    save_path=cfg['models']['cnn_path'],
)

## 4. Training Curves

In [ ]:
plot_training_curves(history, title_suffix='(CNN)')

## 5. Evaluate on Test Set

In [ ]:
model = load_model(PneumoNetCNN(), cfg['models']['cnn_path'], device)
metrics, y_true, y_pred, y_probs = evaluate(model, test_loader, device)
plot_confusion_matrix(y_true, y_pred, classes=test_data.classes)
plot_roc_curve(y_true, y_probs, metrics['roc_auc'])

## 6. Explainability

In [ ]:
sample_img, _ = test_data[0]
show_gradcam(model, sample_img, find_last_conv(model), device)

In [ ]:
test_imgs = torch.stack([test_data[i][0] for i in range(8)])
show_shap(model, test_imgs, device)

In [ ]:
sample_img, _ = test_data[0]
show_lime(model, sample_img.unsqueeze(0), device)

## 7. Gradio Demo
Or from terminal: `python -m src.app --model cnn --weights models/pneumonet_cnn.pth`

In [ ]:
from src.app import load_cnn, launch_app

app_model = load_cnn(cfg['models']['cnn_path'])
launch_app(app_model, title='PneumoNet CNN - Pneumonia Detection')